In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
#import rioxarray
from scipy.stats import percentileofscore
import scipy

In [3]:
# load streamflow data
seasonal_flow_df = pd.read_csv('/nesi/project/niwa00015/queenle/data/flows/seasonal/50yrs/new_full_records.csv')
seasonal_flow_df.date = pd.to_datetime(seasonal_flow_df.date)
seasonal_flow_df.set_index('date', inplace=True)
seasonal_flow_df = seasonal_flow_df.drop(['4901'],axis=1)

# load station mean temperature data
df = pd.read_csv('/home/queenle/hydro_attribution/station_data/temperature_17_series/Combined17SS_02.csv')
df['date'] = df['STATS_YEAR'].astype(str) + '-' + df['STATS_MONTH'].astype(str)
df.index = pd.to_datetime(df['date'])

df_list = []
for site in df['station'].unique():
    ts = df.loc[df['station']==site]['newval']
    ts = ts.rename(site)
    df_list.append(ts)
    
temp_df = pd.concat(df_list, axis=1)
seasonal_temp_df = temp_df.resample('Q-DEC').mean()

# load precipitation station data
stations = ['Blenheim','Christchurch','Dunedin','Gisborne','Hamilton','Hokitika','Invercargill','Milford','Monowai','Napier',
           'Nelson','New Plymouth','Ngakuru','Palmerston North','Paraparaumu','Queenstown','Tauranga','Timaru','Wanganui',
           'Wellington','Whangarei']

df_list = []
for station in stations:
    df = pd.read_csv('/home/queenle/hydro_attribution/station_data/rainfall_21_series/'+station+'.csv')
    df['date'] = df['STATS_YEAR'].astype(str) + '-' + df['STATS_MONTH'].astype(str)
    df.index = pd.to_datetime(df['date'])
    ts = df['obs_value']
    ts = ts.rename(station)
    df_list.append(ts)

precip_df = pd.concat(df_list, axis=1)
seasonal_precip_df = precip_df.resample('Q-DEC').mean()


In [22]:
'''
Save raw obs files
'''
seasons_a = ['autumn','winter','spring','summer']

data_dict = {'streamflow':seasonal_flow_df, 'precip':seasonal_precip_df, 'temp':seasonal_temp_df}

for var in data_dict:
    df = data_dict[var]
    
    for i,s in enumerate([3,6,9,12]):

        # select seasonal time series
        season_df = df.loc[df.index.month == s]

        season_df.to_csv(f'/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/obs/{var}/{var}_obs_{seasons_a[i]}_raw.csv')
        

In [3]:
'''
Use quantile mapping to transform streamflow observation into gaussian distribution
'''

seasons_a = ['autumn','winter','spring','summer']

for i,s in enumerate([3,6,9,12]):
    
    # select seasonal time series
    season_df = seasonal_flow_df.loc[seasonal_flow_df.index.month == s]

    # create container for corrected values
    corrected_df = season_df.copy()
    
    for site in season_df:

        site_ts = season_df[site].values

        corrected = site_ts.copy()
        for j, val in enumerate(site_ts):
            # find percentile of value within the distribution of observations
            if np.isnan(val):
                corrected[j] = val
                continue
                
            percentile = percentileofscore(site_ts[~np.isnan(site_ts)], val, kind= 'mean')
            if percentile == 100 or percentile == 0:
                print('oops')

            # replace with corresponding percentile value in norm distribution
            corrected[j] = scipy.stats.norm.ppf(percentile/100, loc=0, scale=1)

        corrected_df[site] = corrected
    
    corrected_df.to_csv("/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/obs/streamflow/streamflow_obs_"+seasons_a[i]+"_gaussian.csv")
    

In [4]:
'''
Use quantile mapping to transform temperature observation into gaussian distribution
'''

seasons_a = ['autumn','winter','spring','summer']

for i,s in enumerate([3,6,9,12]):
    
    # select seasonal time series
    season_df = seasonal_temp_df.loc[seasonal_temp_df.index.month == s]

    # create container for corrected values
    corrected_df = season_df.copy()
    
    for site in season_df:

        site_ts = season_df[site].values

        corrected = site_ts.copy()
        for j, val in enumerate(site_ts):
            # find percentile of value within the distribution of observations
            if np.isnan(val):
                corrected[j] = val
                continue
                
            percentile = percentileofscore(site_ts[~np.isnan(site_ts)], val, kind= 'mean')
            if percentile == 100 or percentile == 0:
                print('oops')
                
            # replace with corresponding percentile value in norm distribution
            corrected[j] = scipy.stats.norm.ppf(percentile/100, loc=0, scale=1)
            
        corrected_df[site] = corrected
        
    corrected_df.to_csv("/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/obs/temperature/temp_obs_"+seasons_a[i]+"_gaussian.csv")
    

In [5]:
'''
Use quantile mapping to transform precipitation observation into gaussian distribution
'''

seasons_a = ['autumn','winter','spring','summer']

for i,s in enumerate([3,6,9,12]):

    # select seasonal time series
    season_df = seasonal_precip_df.loc[seasonal_precip_df.index.month == s]

    # create container for corrected values
    corrected_df = season_df.copy()
    
    for site in season_df:

        site_ts = season_df[site].values

        corrected = site_ts.copy()
        for j, val in enumerate(site_ts):
            # find percentile of value within the distribution of observations
            if np.isnan(val):
                corrected[j] = val
                continue
                
            percentile = percentileofscore(site_ts[~np.isnan(site_ts)], val, kind= 'mean')
            if percentile == 100 or percentile == 0:
                print('oops')
                
            # replace with corresponding percentile value in norm distribution
            corrected[j] = scipy.stats.norm.ppf(percentile/100, loc=0, scale=1)
            
        corrected_df[site] = corrected
        
    corrected_df.to_csv("/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/obs/precipitation/precip_obs_"+seasons_a[i]+"_gaussian.csv")
    